# Comparative analysis v3

Build the thesis comparison report from completed Common Evaluation outputs. Raw performance is compared only **within the same board size**. Cross-size conclusions use trajectories of controlled within-size contrasts, never a global 8×8-versus-16×16 ranking.

The output is `comparison_report.md` plus CSV/LaTeX exports and seven complementary figures.

In [ ]:
ARCHITECTURE = "Both"   # "Transformer", "Mamba", or "Both"
OBJECTIVE = "Both"       # "AR", "JEPA", or "Both"
BOARD_SIZE = "All"       # 8, 12, 16, or "All"

architecture = ARCHITECTURE.lower()
objective = OBJECTIVE.lower()
board_size = BOARD_SIZE.lower() if isinstance(BOARD_SIZE, str) else int(BOARD_SIZE)
assert architecture in {"transformer", "mamba", "both"}
assert objective in {"ar", "jepa", "both"}
assert board_size in {8, 12, 16, "all"}

## Runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final"
)
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Master_Thesis_Artifacts")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), PROJECT_ROOT
assert ARTIFACTS_ROOT.is_dir(), ARTIFACTS_ROOT
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)
os.chdir(PROJECT_ROOT)

## Metric glossary

In [ ]:
from IPython.display import Markdown, display
from othello_thesis.evaluation.comparison_v3 import metric_glossary_markdown

display(Markdown(metric_glossary_markdown()))

## Build the Markdown report, tables, exports, and figures

In [ ]:
from othello_thesis.evaluation.comparison_v3 import build_comparison_report

comparison = build_comparison_report(
    artifacts_root=ARTIFACTS_ROOT,
    architecture=architecture,
    objective=objective,
    board_size=board_size,
)
print("Markdown:", comparison.report_path)
print("Runs:", len(comparison.runs))
if comparison.missing:
    print("Missing:", *comparison.missing, sep="\n- ")
for warning in comparison.warnings:
    print("WARNING:", warning)
display(Markdown(comparison.report_path.read_text(encoding="utf-8")))

## Figures

In [ ]:
from IPython.display import Image as NotebookImage

for name, path in comparison.figure_paths.items():
    print(name, "->", path)
    display(NotebookImage(filename=str(path)))

print("Exports:")
for name, path in comparison.export_paths.items():
    print(f"- {name}: {path}")